# 혈당코치 우땅이: 챗봇 (LangChain 버전)

예전 챗봇(`챗봇_데모.ipynb`의 10번)과 **같은 기능·같은 답변 규칙**을 LangChain으로 옮긴 노트북이에요.
`master.parquet`가 이미 있어야 해요. (없으면 `챗봇_데모.ipynb`의 8-0 셀을 먼저 돌리세요.)

### 무엇이 바뀌었나
| 부분 | 예전 | 지금 |
|---|---|---|
| 문서 임베딩 | `client.embeddings.create` 직접 호출 | `OpenAIEmbeddings` |
| 벡터 저장·검색 | numpy 배열 + pickle | **FAISS** (`태영_코드/guideline_rag.py`) |
| 도구 설명 | JSON 스키마를 손으로 작성 | `@tool` + pydantic 인자 모델 |
| 도구 호출 루프 | `for` 루프로 직접 구현 | `create_agent` |
| 문서 읽기·청킹·점수 보정 | `guideline_tools.py` | 그대로 (같은 함수를 불러 써요) |
| 안전 필터·출처 footer | 노트북 | 그대로 (에이전트 앞뒤에서 처리) |

빠진 것: 예전 루프의 **"check_food를 여러 번 부르면 meal_sum으로 합치기" 보정**은 에이전트 안에 넣기 까다로워서 뺐어요.
10-6 테스트에서 음식 2개 질문이 `meal_sum`으로 잘 가는지 확인하세요.

### 돌리는 순서
10-0 → 10-1 → 10-2 → (10-3 비교, 선택) → 10-4 → 10-5 → 10-6 테스트 → 10-7 데모

### 먼저 확인
- `혈당코치/mapping2/master.parquet`
- `혈당코치/민석_코드/nutrition_tools.py`
- `혈당코치/태영_코드/guideline_tools.py`, **`혈당코치/태영_코드/guideline_rag.py` (새 파일)**
- `혈당코치/기능2/관련 문서/kdca/*.html`
- 코랩 🔑 Secrets의 `OPENAI_API_KEY` (이 노트북의 접근 토글도 켜기)

### 🛠 민석 수정 (10/22)
예전 노트북(`우땅이혈당선생님`)에서 10/17에 고친 것들이 LangChain 버전에는 빠져 있어서 옮겼어요.
- **안전 필터 오탐 수정** (10-5): "약간 줄여도 돼?", "약과 같이 먹어도 돼?", "의식적으로", "저혈당 예방" 같은 질문이 의료 안내로 막히던 문제
- **푸터 문구** (10-5): "임시 기준" 문구(`nt.NOTICE`)와, 열량·당류·나트륨이 높음일 때 "자체 참고 기준" 안내(`nt.SIDE_NOTICE`)
- **보조 라벨** (10-4, 10-5): `meal_sum` 결과에 열량·당류·나트륨 라벨을 넣고, 높음일 때만 답에 한 줄 언급
- **하루 기록 세션별 분리, 다크 모드 글씨색** (10-7)
- **빈 답변 방지** (10-5)
- **테스트 추가** (10-6): 음식 2개 이상 질문 10개로 `meal_sum` 선택 확인, 안전 필터 오탐 확인
수정한 셀(10-4, 10-5, 10-6, 10-7)은 다시 실행해야 해요. `nutrition_tools.py`는 최신본(`SIDE_NOTICE` 포함)이 드라이브에 있어야 해요.


### 8-0. ✅ 여기부터 돌리세요: master.parquet 만들기

이 셀 하나로 마운트 → 영양DB 읽기 → 검수 파일 반영 → 저장까지 한 번에 해요. (1~2분 걸려요)

돌리기 전 확인
- `mapping2/review_v6_full.xlsx`가 있어야 해요. (민석이 검수를 채워서 올려 둔 파일)
- `검수` 열에 빈칸이나 `?`가 없어야 해요. 남아 있으면 에러가 나고 어떤 행인지 알려줘요.
- `수정_식품코드`가 영양DB에 없는 코드면 에러가 나요. 에러에 나온 행만 고치면 돼요.

검수 파일을 고친 뒤 다시 돌리면 master가 새로 만들어져요(덮어씀).
정상이면 마지막 줄이 `516 개 음식 저장 ... 영양누락: 0`처럼 나와요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, glob, re, unicodedata, shutil, zipfile
import numpy as np, pandas as pd

BASE = "/content/drive/MyDrive/혈당코치"
OUT_DIR = f"{BASE}/mapping2"
PICK_VER = "v6"
LIMIT = None

# 영양DB: 원본 엑셀
NUTRI_PATH = f"{BASE}/식약처_식품영양성분DB_음식.xlsx"
print("영양성분DB:", NUTRI_PATH, f"({os.path.getsize(NUTRI_PATH):,} 바이트)")

# 드라이브에서 바로 읽으면 큰 파일이 깨져 보일 때가 있어서 로컬로 복사한 뒤 검사하고 읽어요
LOCAL = "/content/nutri.xlsx"
shutil.copy(NUTRI_PATH, LOCAL)
assert zipfile.is_zipfile(LOCAL) and zipfile.ZipFile(LOCAL).testzip() is None, \
    "엑셀 파일이 깨져 있어요. 드라이브에서 지우고 원본을 다시 올려 주세요."

# 영양DB 읽기 + food 만들기 (1~2분 걸려요)
raw = pd.read_excel(LOCAL, sheet_name="원본", header=3)
FOOD_GROUPS = ["음식", "농축산물", "수산물"]
PROC_GROUP = "가공식품"
base = raw[raw["DB군"].isin(FOOD_GROUPS) & (raw["상용제품"] == "품목대표")]
proc = raw[raw["DB군"] == PROC_GROUP]
proc_rep = proc[proc["상용제품"] == "품목대표"]
if len(proc_rep) < 100:
    proc_rep = proc.drop_duplicates("식품명")
food = pd.concat([base, proc_rep]).copy()
food["기준"] = np.where(food["DB군"].isin(["음식", PROC_GROUP]), "1인분", "100g")
NUM_COLS = ["1회제공량", "에너지(㎉)", "탄수화물(g)", "총당류(g)", "총 식이섬유(g)", "단백질(g)", "지방(g)", "나트륨(㎎)"]
NUM_COLS = [c for c in NUM_COLS if c in food.columns]
for c in NUM_COLS:
    food[c] = pd.to_numeric(food[c], errors="coerce")
food["식품명"] = food["식품명"].astype(str).str.strip()
food["_key"] = food["식품명"].str.replace(r"\s+", "", regex=True)
food = food.drop_duplicates("식품코드").reset_index(drop=True)
print(len(food), "개 식품")

# 검수 결과 반영 → master
REVIEW_PATH = f"{OUT_DIR}/review_{PICK_VER}_{'test' if LIMIT else 'full'}.xlsx"
rv = pd.read_excel(REVIEW_PATH, dtype=str).fillna("")
rv["검수"] = rv["검수"].str.strip().str.upper()

left = rv[rv["검수"].isin(["", "?"])]
assert left.empty, f"검수가 안 끝난 행이 {len(left)}개 있어요:\n{left['ai_hub_name'].tolist()}"

fix = rv["수정_식품코드"].str.strip()
rv["최종_식품코드"] = np.select([fix != "", rv["검수"] == "X"], [fix, "없음"], default=rv["식품코드"])
rv["근사표시"] = (rv["검수"] == "근사") | ((fix != "") & (rv["수정_관계"].str.strip() == "근사"))

bad = rv[(rv["최종_식품코드"] != "없음") & (~rv["최종_식품코드"].isin(food["식품코드"]))]
assert bad.empty, f"영양DB에 없는 코드가 있어요:\n{bad[['ai_hub_name', '최종_식품코드']]}"

rv["관계"] = np.select(
    [fix != "", rv["검수"] == "근사", rv["검수"] == "O"],
    [np.where(rv["수정_관계"].str.strip() == "근사", "근사", "동일"), "근사", rv["관계"]],
    default=rv["관계"])

ok = (rv[rv["최종_식품코드"] != "없음"][["ai_hub_name", "영문명", "최종_식품코드", "관계", "근사표시"]]
        .rename(columns={"최종_식품코드": "식품코드"}))

KEEP = ["식품코드", "식품명", "식품대분류", "식품상세분류", "기준"] + NUM_COLS
master = ok.merge(food[KEEP], on="식품코드", how="left").reset_index(drop=True)
master.insert(0, "food_id", [f"F{i:04d}" for i in range(1, len(master) + 1)])

factor = np.where(master["기준"] == "100g", 1.0, 100 / master["1회제공량"].replace(0, np.nan))
for c in NUM_COLS[1:]:
    master[c.replace("(", "_100g(")] = master[c] * factor
master["영양누락"] = master[["에너지(㎉)", "탄수화물(g)"]].isna().any(axis=1)

master.to_parquet(f"{OUT_DIR}/master.parquet", index=False)
master.to_csv(f"{OUT_DIR}/master.csv", index=False, encoding="utf-8-sig")
print(len(master), "개 음식 저장 / 관계:", master["관계"].value_counts().to_dict(), "/ 영양누락:", master["영양누락"].sum())

### (참고) 영양누락 확인: 이미 해결했어요
master에서 에너지나 탄수화물이 비어 있는 음식(`영양누락 = True`)을 보는 셀이에요.
처음엔 14개였는데 검수 엑셀에서 코드를 바꿔서 지금은 **0개**예요. 다시 확인하고 싶을 때만 돌리세요.

In [ ]:
m = pd.read_parquet(f"{OUT_DIR}/master.parquet")
print(m[m["영양누락"]][["food_id","ai_hub_name","식품코드","식품명","기준","1회제공량"]].to_string())

### (참고) 영양누락이 다시 생기면 이 셀을 돌리세요
영양누락 음식마다, 후보 중 에너지와 탄수화물 값이 **있는** 것만 보여줘요.
있으면 엑셀의 `수정_식품코드`를 그 코드로 바꾸고 8-0 셀을 다시 돌려요. 없으면 `검수`에 `X`만 적어요.
(영양누락이 0이면 돌릴 필요 없어요.)

In [ ]:
miss = m[m["영양누락"]]["ai_hub_name"].tolist()
for n in miss:
    cand = rv.loc[rv["ai_hub_name"] == n, "후보"].iloc[0].split(" / ")
    codes = [c.split()[0] for c in cand]
    t = food[food["식품코드"].isin(codes)][["식품코드", "식품명", "1회제공량", "에너지(㎉)", "탄수화물(g)"]]
    t = t[t["에너지(㎉)"].notna() & t["탄수화물(g)"].notna()]
    print("■", n)
    print(t.to_string(index=False) if len(t) else "  (값 있는 후보 없음)")

## 민석 파트: 음식 조회 · 혈당 부담 · 한 끼 합산 · 하루 기록

> ✅ 아래 셀을 위에서부터 차례로 돌리세요. (8-0 셀을 먼저 돌려서 master.parquet이 있어야 해요.)

위에서 만든 `master.parquet`을 불러와서 쓰는 부분이에요.
코드 파일: `혈당코치/민석_코드/nutrition_tools.py`

- 음식 이름 조회 (`lookup_text`)
- 혈당 부담 등급 (낮음/보통/높음, 영양성분으로 추정한 참고용이에요. 혈당 수치를 예측하지 않아요)
- 한 끼 합산 (`meal_sum`)
- 하루 식사 기록 (`DailyLog`)

참고: 혈당 부담 기준값(순탄수 15g/40g, 당류 15g)은 **임시**예요. 근거 자료를 찾으면 `nutrition_tools.py`의 `LEVEL_CFG`만 바꾸면 돼요.

#### ✅ 동작 확인: 조회 / 한 끼 합산 / 하루 기록 (돌리세요)
`민석_코드/nutrition_tools.py`를 불러와서 예시로 돌려 보는 셀이에요. 에러 없이 결과가 나오면 정상이에요.

In [ ]:
exec(open("/content/drive/MyDrive/혈당코치/민석_코드/nutrition_tools.py", encoding="utf-8").read())
OUT_DIR = "/content/drive/MyDrive/혈당코치/mapping2"
M = load_master(f"{OUT_DIR}/master.parquet")

# 1) 음식 이름으로 조회
print(lookup_text(M, "마르게리타 피자"))
print(lookup_text(M, "바나나", mult=2))

# 2) 한 끼 합산
ids = [find_food(M, "비빔밥")[0], find_food(M, "미역국")[0]]
r = meal_sum(M, [(i, 1) for i in ids])
print(r["혈당부담"], r["이유"], r["합계"]["탄수화물(g)"])

# 3) 하루 기록
log = DailyLog(M)
log.add("아침", find_food(M, "토스트")[0])
log.add("점심", ids[0])
s = log.summary()
print(s["하루합계"]["혈당부담"], list(s["끼니별"]))

#### (선택) 이상값 확인 ①: 제공량이 작은 브랜드 제품
한 끼 음식인데 `1회제공량`이 100g 미만이면 영양값이 작게 나와 혈당 부담이 낮게 보일 수 있어요. 이런 행이 있으면 검수 엑셀에서 코드를 바꿔요. (지금은 정리 끝)

In [ ]:
sus = M[(M["기준"] == "1인분") & (M["1회제공량"] < 100) & M["식품코드"].str.startswith("P")]
print(len(sus), "개")
print(sus[["food_id", "ai_hub_name", "식품명", "1회제공량", "에너지(㎉)", "탄수화물(g)"]].to_string(index=False))

#### (선택) 이상값 확인 ②: 열량밀도가 너무 낮은 음식
`g당 열량`이 낮은데 탄수화물이 있는 행을 보여줘요. 국, 죽, 주스는 원래 낮아서 정상이에요. 그 외 음식이 나오면 DB 값을 의심해요.

In [ ]:
M = load_master(f"{OUT_DIR}/master.parquet")
d = M[(M["기준"] == "1인분")].copy()
d["kcal_per_g"] = d["에너지(㎉)"] / d["1회제공량"]
sus = d[(d["kcal_per_g"] < 0.5) & (d["탄수화물(g)"] > 5)]
print(len(sus), "개")
print(sus[["food_id","ai_hub_name","식품명","1회제공량","에너지(㎉)","탄수화물(g)","kcal_per_g"]].to_string(index=False))

## 9. 결과 지표 (발표·README용)

> ✅ 8-0 셀을 돌린 뒤에 돌리세요. (검수 수정 건수, 최종 사용 음식 수 등 발표·README용 숫자가 나와요.)

In [ ]:
total = len(rv)
auto_none = (rv["식품코드"] == "없음").sum()
final_none = (rv["최종_식품코드"] == "없음").sum()
changed = (rv["수정_식품코드"].str.strip() != "").sum()

hs = rv[rv["검수그룹"] == "6_높음_검수"]
hs_done = hs[hs["검수"].isin(["O", "X", "근사"])]
err = (hs_done["검수"] == "X").mean() if len(hs_done) else float("nan")

print(f"전체 카테고리: {total}")
print(f"LLM 자동 매칭: {total - auto_none}  /  '없음': {auto_none}")
print(f"사람이 수정한 건수: {changed}")
print(f"최종 사용 음식: {total - final_none}")
print(f"확신도 '높음' 무작위 {len(hs_done)}개 검수 오류율: {err:.1%}")
print(rv["확신도"].value_counts())

## 10. 챗봇: 기능 1·2·3·4 통합

> ✅ 10-0부터 차례로 돌리세요. 위 1~9번과 상관없이 이 섹션만 돌려도 돼요. (master.parquet만 있으면 돼요)


기능별 코드는 `.py` 파일로 나눠 두고, 이 노트북은 **불러와서 LLM 도구로 등록하고 대화만** 담당해요.

| 기능 | 담당 | 파일 | LLM 도구 |
|---|---|---|---|
| 1. 식사 방법 물어보기 (문서 근거 + 조건·대체 메뉴) | 태영 · 민석 | `태영_코드/guideline_tools.py`, `민석_코드/nutrition_tools.py` | `search_guideline`, `search_menu`, `find_alternatives` |
| 2. 혈당 부담 확인 | 민석 | `민석_코드/nutrition_tools.py` | `check_food` |
| 3. 한 끼 합산 평가 | 민석 | `민석_코드/nutrition_tools.py` | `meal_sum` |
| 4. 하루 식사 기록 | 민석 | `민석_코드/nutrition_tools.py` (`DailyLog`) | `record_meal`, `daily_summary`, `remove_record` |

> 하루 기록은 이 노트북이 켜져 있는 동안만 유지돼요. 데모의 **새 대화** 버튼을 누르면 기록도 비워져요.

**팀 약속**
1. 각 `.py`의 함수는 **dict를 반환**해요 (LLM에 JSON으로 넘겨요).
2. API 키, 시스템 프롬프트, 안전 필터, 화면(Gradio)은 **이 노트북에만** 둬요.
3. `.py`를 고친 뒤에는 10-2 셀(불러오기)부터 다시 실행하면 돼요. (`importlib.reload`)

**순서**: 10-0 → 10-1 → 10-2 → 10-3 → 10-4 → (10-5 테스트) → 10-6 데모

### 10-0. 설치

In [ ]:
!pip -q install -U langchain langchain-openai langchain-community faiss-cpu beautifulsoup4 pypdf gradio

### 10-1. 설정
LLM과 임베딩 모델을 LangChain 객체로 만들어요. `client`는 10-3에서 예전 검색과 비교할 때만 써요.

In [ ]:
import os, sys, re, json, importlib, warnings
import pandas as pd
from google.colab import drive, userdata
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# langchain-community(FAISS가 들어 있는 패키지)가 곧 정리될 예정이라는 안내 경고예요. 동작에는 문제없어요.
warnings.filterwarnings('ignore', message='.*langchain-community.*')

drive.mount('/content/drive')   # 이미 연결돼 있으면 그대로 둬요

BASE        = '/content/drive/MyDrive/혈당코치'
TOOL_DIRS   = [f'{BASE}/민석_코드', f'{BASE}/태영_코드']     # 각자 .py를 둔 폴더
MASTER_PATH = f'{BASE}/mapping2/master.parquet'
DOCS_DIR    = f'{BASE}/기능2/관련 문서'                     # 이 안의 kdca/ 폴더에 3388.html 등

OPENAI_KEY  = userdata.get('OPENAI_API_KEY')
CHAT_MODEL  = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'

llm        = ChatOpenAI(model=CHAT_MODEL, temperature=0.2, api_key=OPENAI_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL, api_key=OPENAI_KEY)
client     = OpenAI(api_key=OPENAI_KEY)    # 10-3 비교용

### 10-2. 기능 모듈 불러오기 + FAISS 인덱스
처음 한 번은 문서를 임베딩해서 `DOCS_DIR/faiss_index/`에 저장해요. 다음부터는 저장된 인덱스를 불러와요.
문서·청킹·임베딩 모델이 바뀌면 자동으로 다시 만들어요.

In [ ]:
for d in TOOL_DIRS:
    if d not in sys.path:
        sys.path.append(d)

import nutrition_tools as nt      # 민석: 조회·혈당 부담·메뉴·한 끼 합산·하루 기록
import guideline_tools as gt      # 태영: 문서 읽기·청킹 (예전 검색 코드도 들어 있어요)
import guideline_rag as grag      # 태영: LangChain + FAISS 검색
importlib.reload(nt); importlib.reload(gt); importlib.reload(grag)   # .py를 고쳤으면 이 셀만 다시 실행

M = nt.load_master(MASTER_PATH)
print(f'master: {len(M)}개 음식')

rag = grag.GuidelineRAG(DOCS_DIR, embeddings).build()

### 10-3. (선택) 예전 검색과 결과 비교
같은 청크·같은 임베딩 모델·같은 점수 보정이라 **상위 문서가 같아야 정상**이에요.
점수가 소수점 셋째 자리에서 조금 다를 수는 있어요. 크게 다르면 옮기는 과정에서 뭔가 빠진 거예요.
(예전 인덱스는 `guideline_index.pkl` 캐시를 쓰니까 문서 임베딩 비용은 안 들어요.)

In [ ]:
old = gt.GuidelineIndex(DOCS_DIR, client, embed_model=EMBED_MODEL).build(verbose=False)

CHECK = [
    ('외식할 때 주의할 점',       ['외식'],     '일반'),
    ('밥 먹는 순서',              ['먹는방법'], '일반'),
    ('임신 중 간식 먹는 법',       ['간식'],     '임산부'),
    ('술 마실 때 안주 고르기',     ['음주'],     '일반'),
    ('영양표시에서 당류 보는 법',  ['영양표시'], '일반'),
    ('체중 줄일 때 식사량',        [],           '체중조절'),
]

def _top(hits):
    return f"{hits[0]['문서']} > {hits[0]['섹션']} ({hits[0]['score']})" if hits else '-'

rows = []
for q, sit, tgt in CHECK:
    a = old.search(q, sit, tgt)['결과']
    b = rag.search(q, sit, tgt)['결과']
    key = lambda h: (h['문서'], h['내용'][:40])
    same = len({key(h) for h in a} & {key(h) for h in b})
    rows.append({'질문': q, '예전 1위': _top(a), 'LangChain 1위': _top(b),
                 '상위 4개 겹침': f'{same}/{max(len(a), len(b), 1)}'})
pd.set_option('display.max_colwidth', 80)
pd.DataFrame(rows)

### 10-4. LangChain 도구 등록

- `tool_*` 함수(실제 동작)는 예전과 같아요. `search_guideline`만 FAISS 검색(`rag.search`)으로 바뀌었어요.
- LLM이 보는 도구 설명은 `@tool`의 docstring, 인자 설명은 pydantic `Field(description=...)`에서 만들어져요.
- 도구가 돌려준 결과는 `_USED`에 모아 두고, 답변 아래 출처 footer를 만들 때 써요.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain.tools import tool

FOOD_SOURCE = '식약처 식품영양성분DB'
LOG   = nt.DailyLog(M)
MEALS = nt.DailyLog.MEALS          # ['아침', '점심', '저녁', '간식']

# ───────── 실제 동작 (예전과 같음) ─────────
def _foods(items):
    """LLM이 넘긴 음식 목록을 [{'food': ..., 'amount': ...}] 형태로 맞춰요."""
    out = []
    for f in items or []:
        if hasattr(f, 'model_dump'):
            f = f.model_dump()
        out.append(f if isinstance(f, dict) else {'food': str(f), 'amount': 1})
    return out

def tool_check_food(food, amount=1.0):
    ids = nt.find_food(M, food)
    if not ids:
        return {'결과': None, '안내': f"'{food}'은(는) 데이터에 없는 음식이에요."}
    d = nt.lookup(M, ids[0], float(amount))
    d['비슷한 이름'] = [M.loc[i, 'ai_hub_name'] for i in ids[1:4]]
    return {'결과': d, '출처': FOOD_SOURCE, '참고': nt.NOTICE}

def tool_search_menu(kinds=None, max_carb=None, max_kcal=None, min_protein=None, sort='탄수화물', n=5, cuisine=None):
    res = nt.search_foods(M, kinds, max_carb, max_kcal, min_protein, sort, int(n), cuisine=cuisine)
    relaxed = False
    if not res and (max_carb or max_kcal or min_protein):      # 숫자 조건을 풀고 가장 가까운 메뉴
        res = nt.search_foods(M, kinds, None, None, None, sort, int(n), cuisine=cuisine)
        relaxed = bool(res)
    if not res:
        return {'결과': [], '안내': '조건에 맞는 메뉴를 찾지 못했어요.'}
    return {'결과': res, '숫자조건_완화': relaxed, '출처': FOOD_SOURCE, '참고': nt.NOTICE}

def tool_find_alternatives(food, n=5):
    base, alts = nt.find_alternatives(M, food, int(n))
    if base is None:
        return {'결과': None, '안내': f"'{food}'은(는) 데이터에 없는 음식이에요."}
    return {'기준음식': base, '결과': alts, '출처': FOOD_SOURCE, '참고': nt.NOTICE,
            **({} if alts else {'안내': '같은 분류에서 탄수화물이 더 적은 메뉴를 찾지 못했어요.'})}

def tool_search_guideline(query, situations=None, target='일반'):
    return rag.search(query, situations, target)          # ★ 바뀐 곳: FAISS 검색

def tool_meal_sum(foods):
    items, found, missing = [], [], []
    for f in _foods(foods):
        name, amount = f.get('food'), float(f.get('amount', 1) or 1)
        ids = nt.find_food(M, name)
        if ids:
            items.append((ids[0], amount)); found.append(name)
        else:
            missing.append(name)
    if not items:
        return {'결과': None, '안내': f"{', '.join(missing)}은(는) 데이터에 없는 음식이에요."}
    r = nt.meal_sum(M, items)
    rows = [{'입력': q, '음식': d['음식'], '기준': d['기준'], '배수': d['배수'],
             '열량_kcal': round(d['영양']['에너지(㎉)']), '탄수화물_g': round(d['영양']['탄수화물(g)'], 1),
             '순탄수_g': round(d['영양']['순탄수(g)'], 1), '혈당부담': d['혈당부담'], '근사': d['근사']}
            for q, d in zip(found, r['항목'])]
    top = max(rows, key=lambda x: x['순탄수_g'])
    tot = r['합계']
    return {'결과': {
                '항목': rows,
                '합계': {'열량_kcal': round(tot['에너지(㎉)']), '탄수화물_g': round(tot['탄수화물(g)'], 1),
                        '순탄수_g': round(tot['순탄수(g)'], 1), '당류_g': round(tot['당류(g)'], 1)},
                '한끼_혈당부담': r['혈당부담'], '이유': r['이유'],
                '가장_부담큰_음식': top['음식'],
                '가장_부담큰_음식_비중': f"{top['순탄수_g'] / max(tot['순탄수(g)'], 1e-9):.0%}",
                '보조': {k: [v[0], round(v[1]), v[2]] for k, v in r['보조'].items()},
            },
            '참고': r['참고'], '데이터에_없는_음식': missing, '출처': FOOD_SOURCE}

def _meal_brief(r):
    t = r['합계']
    return {'음식': [d['음식'] for d in r['항목']], '열량_kcal': round(t['에너지(㎉)']),
            '탄수화물_g': round(t['탄수화물(g)'], 1), '순탄수_g': round(t['순탄수(g)'], 1),
            '당류_g': round(t['당류(g)'], 1), '혈당부담': r['혈당부담']}

def tool_daily_summary():
    if not LOG.items:
        return {'결과': None, '안내': '아직 오늘 기록한 식사가 없어요.'}
    s = LOG.summary()
    day = s['하루합계']['합계']
    return {'결과': {
                '끼니별': {meal: _meal_brief(r) for meal, r in s['끼니별'].items()},
                '하루합계': {'열량_kcal': round(day['에너지(㎉)']), '탄수화물_g': round(day['탄수화물(g)'], 1),
                          '순탄수_g': round(day['순탄수(g)'], 1), '당류_g': round(day['당류(g)'], 1),
                          '나트륨_mg': round(day['나트륨(㎎)'])},
                '기록_수': len(LOG.items)},
            '안내': '하루 합계는 숫자만 보여줘요. 혈당 부담 등급은 끼니별로 판단해요.',
            '출처': FOOD_SOURCE}

def tool_record_meal(meal, foods):
    if meal not in MEALS:
        return {'결과': None, '안내': f'끼니는 {MEALS} 중 하나로 알려 주세요.'}
    added, missing = [], []
    for f in _foods(foods):
        name, amount = f.get('food'), float(f.get('amount', 1) or 1)
        ids = nt.find_food(M, name)
        if ids:
            LOG.add(meal, ids[0], amount); added.append(M.loc[ids[0], 'ai_hub_name'])
        else:
            missing.append(name)
    out = tool_daily_summary()
    out['이번에_기록'] = {'끼니': meal, '음식': added}
    if missing:
        out['데이터에_없는_음식'] = missing
    return out

def tool_remove_record(meal=None, food=None):
    removed = []
    for i in range(len(LOG.items) - 1, -1, -1):
        m, fid, _ = LOG.items[i]
        name = M.loc[fid, 'ai_hub_name']
        if (meal is None or m == meal) and (food is None or food.replace(' ', '') in name.replace(' ', '')):
            LOG.remove(i); removed.append(f'{m} {name}')
    out = tool_daily_summary()
    out['삭제'] = removed or '지울 기록을 찾지 못했어요.'
    return out

# ───────── 실행 기록: 출처 footer와 테스트에 써요 ─────────
_USED = []   # [(도구 이름, 결과 dict), ...]  chat()이 질문마다 비워요

def _run(name, fn, **kwargs):
    try:
        result = fn(**kwargs)
    except Exception as e:
        result = {'오류': f'{type(e).__name__}: {e}'}
    _USED.append((name, result))
    return json.dumps(result, ensure_ascii=False, default=str)   # LLM에는 JSON 문자열로 넘겨요

# ───────── LLM이 보는 인자 설명 ─────────
Situation = Literal[tuple(gt.SITUATION_KEYWORDS)]   # '외식', '간식', '음주', ...
Target    = Literal[tuple(gt.TARGETS)]              # '일반', '임산부', '체중조절'
Meal      = Literal[tuple(MEALS)]                   # '아침', '점심', '저녁', '간식'

class FoodItem(BaseModel):
    food: str = Field(description='음식 이름')
    amount: float = Field(1.0, description='먹는 양 배수. 기본 1, 반 공기면 0.5, 곱빼기면 1.5')

class GuidelineArgs(BaseModel):
    query: str = Field(description='검색할 내용 (예: 중식 외식 시 주의할 점)')
    situations: Optional[list[Situation]] = Field(None, description='질문에 해당하는 상황')
    target: Target = Field('일반', description='임신 중이면 임산부, 체중 감량 목적이면 체중조절')

class CheckFoodArgs(BaseModel):
    food: str = Field(description='음식 이름 (예: 짜장면)')
    amount: float = Field(1.0, description='먹는 양 배수. 기본 1, 반만 먹으면 0.5, 곱빼기면 1.5')

class SearchMenuArgs(BaseModel):
    kinds: Optional[list[str]] = Field(None, description='음식 이름·분류에 들어갈 짧은 단어 (예: 국물 요리 → ["국","탕","찌개","전골"], 면 → ["면","국수"], 구이·찜 → ["구이","찜"])')
    max_carb: Optional[float] = Field(None, description='탄수화물 상한(g). 숫자가 없으면 넣지 않는다')
    max_kcal: Optional[float] = Field(None, description='열량 상한(kcal)')
    min_protein: Optional[float] = Field(None, description='단백질 하한(g)')
    sort: Literal['탄수화물', '열량', '순탄수'] = Field('탄수화물', description='정렬 기준')
    n: int = Field(5, description='개수, 기본 5')
    cuisine: Optional[Literal['한식', '중식', '일식', '양식', '분식']] = Field(None, description='요리 종류. "중식집", "일식당"처럼 식당·요리 종류를 말했을 때만 넣는다')

class AlternativesArgs(BaseModel):
    food: str = Field(description='바꾸고 싶은 음식 이름')
    n: int = Field(5, description='개수, 기본 5')

class MealSumArgs(BaseModel):
    foods: list[FoodItem] = Field(description='한 끼에 먹는 음식 목록')

class RecordMealArgs(BaseModel):
    meal: Meal = Field(description='끼니')
    foods: list[FoodItem] = Field(description='먹은 음식 목록')

class RemoveRecordArgs(BaseModel):
    meal: Optional[Meal] = Field(None, description='지울 끼니 (없으면 모든 끼니에서 찾음)')
    food: Optional[str] = Field(None, description='지울 음식 이름 (없으면 그 끼니 전체)')

# ───────── LangChain 도구 (docstring = LLM이 보는 도구 설명) ─────────
@tool('search_guideline', args_schema=GuidelineArgs)
def lc_search_guideline(query: str, situations: Optional[list] = None, target: str = '일반') -> str:
    """[기능 1] 당뇨 식사요법 공식 문서(질병관리청 등)에서 먹는 방법, 상황별 요령(외식·간식·음주 등), 원칙과 이유를 검색한다. "어떻게 먹어?", "뭘 조심해?" 같은 식사 방법 질문에는 항상 호출한다."""
    return _run('search_guideline', tool_search_guideline, query=query, situations=situations, target=target)

@tool('check_food', args_schema=CheckFoodArgs)
def lc_check_food(food: str, amount: float = 1.0) -> str:
    """[기능 2] 음식 **딱 하나**의 영양(열량·탄수화물·식이섬유·당류)과 혈당 부담 등급(낮음/보통/높음)을 조회한다. "짜장면 먹어도 돼?", "라면 칼로리 얼마야?"처럼 음식이 하나일 때만 쓴다. 질문에 음식이 2개 이상 나오면 이 도구를 쓰지 말고 meal_sum을 쓴다."""
    return _run('check_food', tool_check_food, food=food, amount=amount)

@tool('search_menu', args_schema=SearchMenuArgs)
def lc_search_menu(kinds: Optional[list] = None, max_carb: Optional[float] = None, max_kcal: Optional[float] = None,
                   min_protein: Optional[float] = None, sort: str = '탄수화물', n: int = 5,
                   cuisine: Optional[str] = None) -> str:
    """[기능 1] 조건에 맞는 메뉴를 탄수화물(또는 열량) 낮은 순으로 찾는다. "탄수화물 적은 국물 요리 추천해줘" 같은 질문, 또는 식사요법 원칙(예: 튀김보다 구이·찜)을 실제 메뉴 예시로 보여줄 때 쓴다."""
    return _run('search_menu', tool_search_menu, kinds=kinds, max_carb=max_carb, max_kcal=max_kcal,
                min_protein=min_protein, sort=sort, n=n, cuisine=cuisine)

@tool('find_alternatives', args_schema=AlternativesArgs)
def lc_find_alternatives(food: str, n: int = 5) -> str:
    """[기능 1] 어떤 음식 대신 먹을 만한, 같은 분류에서 탄수화물이 더 적은 메뉴를 찾는다. "짜장면 대신 뭐 먹지?" 같은 질문에 쓴다."""
    return _run('find_alternatives', tool_find_alternatives, food=food, n=n)

@tool('meal_sum', args_schema=MealSumArgs)
def lc_meal_sum(foods: list) -> str:
    """[기능 3] 음식이 2개 이상 나오는 질문이면 표현("혈당 부담 어때?", "먹어도 돼?", "칼로리 얼마야?")과 상관없이 이 도구를 쓴다. 여러 음식을 합산해서 열량·탄수화물 합계, 한 끼 혈당 부담 등급, 부담을 가장 많이 만드는 음식을 알려준다. "점심에 밥, 김치찌개, 계란말이 먹을 거야" 같은 질문에 쓴다. 이때는 check_food를 음식마다 따로 부르지 말고 이 도구 하나로 처리한다."""
    return _run('meal_sum', tool_meal_sum, foods=foods)

@tool('record_meal', args_schema=RecordMealArgs)
def lc_record_meal(meal: str, foods: list) -> str:
    """[기능 4] 이미 먹은 음식을 하루 기록에 끼니별로 저장한다. "아침에 토스트 먹었어", "점심 먹은 거 기록해줘"처럼 먹었다고 말하거나 기록을 부탁할 때 쓴다. 양을 말하지 않았으면 amount 1(1인분)로 바로 기록한다. 양을 되묻지 않는다. 저장 후 오늘 기록 요약을 함께 돌려준다."""
    return _run('record_meal', tool_record_meal, meal=meal, foods=foods)

@tool('daily_summary')
def lc_daily_summary() -> str:
    """[기능 4] 오늘 기록한 식사를 끼니별로 정리하고 하루 열량·탄수화물·당류 합계를 보여준다. "오늘 먹은 거 정리해줘", "오늘 탄수화물 얼마나 먹었어?" 같은 질문에 쓴다."""
    return _run('daily_summary', tool_daily_summary)

@tool('remove_record', args_schema=RemoveRecordArgs)
def lc_remove_record(meal: Optional[str] = None, food: Optional[str] = None) -> str:
    """[기능 4] 하루 기록에서 잘못 넣은 음식을 지운다. "점심 기록 지워줘", "아침에 우유 빼줘" 같은 요청에 쓴다."""
    return _run('remove_record', tool_remove_record, meal=meal, food=food)

TOOLS = [lc_search_guideline, lc_check_food, lc_search_menu, lc_find_alternatives,
         lc_meal_sum, lc_record_meal, lc_daily_summary, lc_remove_record]

# 도구만 따로 확인 (LLM 없이). LangChain 도구는 .invoke({인자})로 불러요.
print([t.name for t in TOOLS])
print(lc_check_food.invoke({'food': '짜장면'})[:200])
print(lc_meal_sum.invoke({'foods': [{'food': '밥'}, {'food': '김치찌개'}, {'food': '계란말이'}]})[:200])
print(lc_record_meal.invoke({'meal': '아침', 'foods': [{'food': '토스트'}]})[:200]); LOG.items.clear()
print([h['문서'] for h in json.loads(lc_search_guideline.invoke({'query': '외식할 때 주의할 점', 'situations': ['외식']}))['결과']])
_USED.clear()

### 10-5. 에이전트 + 안전 필터 + 챗봇

- 예전의 직접 짠 도구 호출 루프 대신 `create_agent`가 "LLM 호출 → 도구 실행 → 다시 LLM"을 반복해요.
- **안전 필터**는 에이전트를 부르기 **전**에, **출처 footer**는 에이전트가 끝난 **후**에 코드가 처리해요. (예전과 같음)
- `max_steps`는 LangChain의 `recursion_limit`으로 바꿔서 넘겨요. 너무 길어지면 멈추고 안내 문구를 돌려줘요.

In [ ]:
from langchain.agents import create_agent
from langgraph.errors import GraphRecursionError

SAFETY_RULES = [
    (r'저혈당.{0,8}(증상|의심|왔|온|같|때|인데|이야|이에요)|식은땀|손.{0,4}떨리|어지러|의식(이|을)?\s*(없|흐|잃)|쓰러',
     '⚠️ 저혈당이 의심되면 바로 당분(사탕, 주스 등 15~20g)을 섭취하고 15분 뒤 다시 확인하세요. '
     '증상이 계속되거나 의식이 흐려지면 즉시 119에 연락하세요. 평소 저혈당 대처법은 담당 의료진과 미리 정해 두는 것이 안전해요.'),
    (r'인슐린|단위.{0,4}(맞|주사)|(당뇨약|혈당약|약물|처방약|먹는 약)|약(을|은|이)?\s*(용량|조절|끊|줄이|줄여|늘리|늘려)|메트포르민|복용량',
     '약이나 인슐린 용량은 개인 상태에 따라 달라서 제가 안내해 드릴 수 없어요. 담당 의사나 약사와 상담해 주세요.'),
    (r'(당뇨약|혈당약|약물|처방약|먹는 약).{0,10}(같이|함께|동시).{0,6}(먹|드|복용)|자몽.{0,8}(약|복용)|(당뇨약|혈당약|약물|처방약|복용).{0,8}자몽',
     '약과 음식의 상호작용은 복용 중인 약에 따라 달라요. 약사나 담당 의사에게 확인해 주세요.'),
]

def safety_check(text):
    for pat, msg in SAFETY_RULES:
        if re.search(pat, text):
            return msg
    return None

SYSTEM_PROMPT = '''너는 당뇨 관리 앱 '혈당코치'의 식사 도우미 캐릭터 '우땅이'야. 친근하고 다정한 존댓말을 쓰고, 자기소개는 사용자가 인사하거나 누구냐고 물을 때만 하고, 평소 답변은 인사 없이 바로 본론부터 말해. 도구를 골라 쓰고, 도구 결과만 근거로 답해.

도구 고르기:
- "어떻게 먹어?", "뭘 조심해?" 같은 식사 방법 → search_guideline
- 음식 **하나**에 대해 "먹어도 돼?", "칼로리·탄수화물 얼마야?", "혈당 많이 올라?" → check_food
- 음식이 **2개 이상** 나오면 표현과 상관없이 → meal_sum (check_food를 여러 번 부르지 않는다)
  예: "저녁에 김밥이랑 라면 먹으면 혈당 부담 어때?" → meal_sum
- "~ 추천해줘", 조건(탄수화물 ○g 이하 등)이 있는 메뉴 찾기 → search_menu
  중식집·일식당처럼 요리 종류를 말하면 search_menu의 cuisine에 넣는다.
- "~ 대신 뭐 먹지?" → find_alternatives
- 한 끼에 음식 여러 개를 먹을 **예정**이면(예: "밥, 김치찌개, 계란말이 먹을 거야") → meal_sum 하나로 처리
- 이미 **먹었다**고 하거나 기록을 부탁하면(예: "아침에 토스트 먹었어", "기록해줘") → record_meal
  양을 말하지 않았으면 1인분으로 **바로 기록**한다. 양을 되묻지 않는다.
  끼니(아침/점심/저녁/간식)를 알 수 없을 때만 어느 끼니인지 물어본다.
- "오늘 먹은 거 정리해줘", "오늘 탄수화물 얼마나 먹었어?" → daily_summary
- "~ 기록 지워줘", "~ 빼줘" → remove_record
- 질문에 여러 의도가 섞이면 도구를 여러 개 부른다.
  예: "중식집 가는데 뭐 시키는 게 나아?" → search_guideline + search_menu(cuisine="중식")
  예: "라면 먹고 싶은데 어떻게 먹어야 해?" → check_food + search_guideline
  예: "점심에 비빔밥이랑 된장국 먹을 건데 어떻게 먹으면 좋아?" → meal_sum + search_guideline

도구 선택 주의:
- 음식을 2개 이상 말하면 (예: "햄버거랑 감자튀김 먹어도 돼?", "A랑 B 같이 먹으면") check_food를 음식마다 따로 부르지 말고, 반드시 meal_sum 하나에 모두 담아 한 번만 부른다. check_food는 음식이 1개일 때만 쓴다.

답변 규칙:
1. 숫자(열량, 탄수화물, 당류 등)는 도구 결과에 있는 값만 쓰고, 기준(1인분 ○g 또는 100g)을 함께 밝힌다.
2. 먹는 방법과 이유는 search_guideline 결과에 있는 내용만 말한다. 결과가 없으면 "공식 자료에서 찾지 못했어요"라고 한다.
3. 혈당 부담 등급은 참고용 추정이며, 혈당 수치를 예측하지 않는다.
4. 결과에 '근사'가 true면 "가장 가까운 음식 기준"이라고 밝힌다.
5. meal_sum 결과를 말할 때는 합계(열량, 탄수화물), 한 끼 혈당 부담 등급, 부담을 가장 많이 만드는 음식을 꼭 말한다. 데이터에 없는 음식이 있으면 그것도 알려준다.
6. 하루 기록을 말할 때는 끼니별 음식과 혈당 부담, 하루 합계(열량, 탄수화물, 당류)를 정리한다. 하루 합계에는 혈당 부담 등급을 붙이지 않는다.
7. 약, 인슐린, 진단, 치료는 다루지 않고 의료진 상담을 권한다.
8. check_food나 meal_sum 결과의 '보조'(열량·당류·나트륨)는 '높음'인 항목이 있을 때만 한 문장으로 짧게 언급한다. (예: "나트륨이 높음 구간이에요")
9. 존댓말로 3~6문장. 메뉴 목록은 줄바꿈해서 보여준다. 출처와 참고 문구는 직접 쓰지 않는다 (시스템이 붙인다).
'''

agent = create_agent(llm, tools=TOOLS, system_prompt=SYSTEM_PROMPT)

def _side_high(result):
    r = result.get('결과')
    if isinstance(r, dict):
        side = r.get('보조') or {}
        return any(v and v[0] == '높음' for v in side.values())
    return False

def make_footer(used):
    docs, db, burden, side = [], False, False, False
    for name, result in used:
        if name == 'search_guideline':
            docs += [s for s in grag.sources_of(result) if s not in docs]
        elif result.get('결과'):
            db, burden = True, True
            if name in ('check_food', 'meal_sum') and _side_high(result):
                side = True
    labels = docs[:3] + ([FOOD_SOURCE] if db else [])
    lines = [f"(출처 : {', '.join(labels)})"] if labels else []
    if burden:
        lines.append(f'※ {nt.NOTICE}')
    if side:
        lines.append(f'※ {nt.SIDE_NOTICE}')
    return '\n'.join(lines)

def _content_text(content):
    # 모델에 따라 content가 문자열 또는 블록 리스트로 와요
    if isinstance(content, str):
        return content
    return ''.join(b.get('text', '') for b in content if isinstance(b, dict))

def chat(question, history=None, verbose=False, max_steps=5):
    """예전 chat()과 같은 입력·출력: {'answer', 'tools', 'footer'}"""
    blocked = safety_check(question)
    if blocked:
        return {'answer': blocked, 'tools': [], 'footer': ''}

    _USED.clear()
    messages = list(history or []) + [{'role': 'user', 'content': question}]
    try:
        # LLM 호출 1번 + 도구 실행 1번 = 2단계. max_steps번 돌고 마지막 답까지 허용
        out = agent.invoke({'messages': messages}, config={'recursion_limit': 2 * max_steps + 1})
        answer = _content_text(out['messages'][-1].content).strip()
        if verbose:
            for m in out['messages']:
                for tc in getattr(m, 'tool_calls', None) or []:
                    print(f"🔧 {tc['name']}({tc['args']})")
    except GraphRecursionError:
        answer = '질문을 처리하는 단계가 너무 길어졌어요. 조금 나눠서 다시 물어봐 주세요.'

    answer = answer or '답변을 만들지 못했어요. 다시 한 번 물어봐 주세요.'
    used = list(_USED)
    footer = make_footer(used)
    return {'answer': f'{answer}\n{footer}'.strip(), 'tools': [n for n, _ in used], 'footer': footer}

print(chat('중식집 가는데 뭐 시키는 게 나아?', verbose=True)['answer'])

### 10-6. 테스트 질문 세트 (도구를 제대로 고르는지)
예전 노트북과 같은 질문이에요. **예전 결과(100%)와 비교**해서 LangChain으로 옮기면서 달라진 게 없는지 확인해요.
특히 9·10번(음식 2개 이상)은 예전엔 코드 보정이 있었는데 지금은 없으니 `meal_sum`으로 가는지 봐 주세요.

In [ ]:
TEST_SET = [
    ('외식할 때 뭘 조심해야 해?',                 {'search_guideline'}),
    ('중식집 가는데 뭐 시키는 게 나아?',           {'search_guideline', 'search_menu'}),
    ('라면 먹고 싶은데 어떻게 먹어야 해?',         {'check_food', 'search_guideline'}),
    ('짜장면 먹어도 돼?',                         {'check_food'}),
    ('탄수화물 적은 국물 요리 추천해줘',           {'search_menu'}),
    ('짜장면 대신 먹을 만한 거 있어?',             {'find_alternatives'}),
    ('밥 먹을 때 어떤 순서로 먹어야 해?',          {'search_guideline'}),
    ('임신성 당뇨인데 간식은 어떻게 먹어야 해?',   {'search_guideline'}),
    ('점심에 밥, 김치찌개, 계란말이 먹을 거야',    {'meal_sum'}),
    ('저녁에 김밥이랑 라면 먹으면 혈당 부담 어때?', {'meal_sum'}),
    ('아침에 토스트랑 우유 먹었어. 기록해줘',      {'record_meal'}),
    ('오늘 먹은 거 정리해줘',                     {'daily_summary'}),
    ('짜장면 먹으면 인슐린 몇 단위 맞아야 해?',    set()),
    ('손이 떨리고 식은땀 나는데 뭐 먹어?',         set()),
]

rows = []
for q, expected in TEST_SET:
    out = chat(q)
    got = set(out['tools'])
    rows.append({'질문': q, '기대': ', '.join(sorted(expected)) or '(안전 응답)',
                 '실제': ', '.join(sorted(got)) or '(안전 응답)',
                 '도구 일치': expected <= got if expected else not got,
                 'check_food 여러 번': out['tools'].count('check_food') >= 2,
                 '답변': out['answer']})
LOG.items.clear()   # 테스트로 넣은 기록 비우기
result_df = pd.DataFrame(rows)
print(f"도구 일치율 {result_df['도구 일치'].mean():.0%}")
pd.set_option('display.max_colwidth', 200)
result_df

# ───────── 추가 테스트 1: 음식 2개 이상 질문이 meal_sum 하나로 가는지 (check_food 여러 번 보정이 없으므로 확인용) ─────────
MULTI = [
    '밥이랑 된장찌개 먹으면 혈당 부담 어때?', '떡볶이랑 순대 같이 먹어도 돼?',
    '아침으로 토스트랑 우유 먹을 건데 괜찮아?', '라면이랑 김밥 칼로리 얼마야?',
    '비빔밥이랑 된장국 먹으면 혈당 많이 올라?', '점심으로 칼국수랑 만두 먹을래',
    '햄버거랑 감자튀김 먹어도 돼?', '짜장면이랑 짬뽕 둘 다 먹으면 어때?',
    '김치찌개랑 계란말이, 밥 먹어도 돼?', '냉면이랑 갈비탕 같이 먹으면 혈당 부담 어때?',
]
mrows = []
for q in MULTI:
    t = chat(q)['tools']
    mrows.append({'질문': q, '실제': ', '.join(t), 'meal_sum만 사용': ('meal_sum' in t and 'check_food' not in t)})
LOG.items.clear()
mdf = pd.DataFrame(mrows)
print(f"음식 2개 이상 질문 {len(MULTI)}개 중 meal_sum만 쓴 비율: {mdf['meal_sum만 사용'].mean():.0%}")
display(mdf)

# ───────── 추가 테스트 2: 안전 필터 오탐 확인 (LLM 호출 없음) ─────────
SAFE_OK = ['밥 양을 약간 줄여도 돼?', '점심에 밥 약간 늘려도 돼?', '약과 한 개 같이 먹어도 돼?',
           '의식적으로 천천히 먹으면 혈당에 도움 돼?', '저혈당 예방하려면 간식 뭐 먹어?']
SAFE_BLOCK = ['짜장면 먹으면 인슐린 몇 단위 맞아야 해?', '저혈당 온 것 같은데 뭐 먹어?', '당뇨약 줄여도 돼?',
              '혈당약이랑 같이 먹어도 돼?', '어지러운데 뭐 먹어?']
bad = [q for q in SAFE_OK if safety_check(q)] + [q for q in SAFE_BLOCK if not safety_check(q)]
print('안전 필터 오작동:', bad or '없음')


### 10-7. 데모 (Gradio)
예전과 같은 화면이에요. 다크 모드 글씨색 수정과 접속자별 하루 기록(`gr.State`)이 들어갔어요.

In [ ]:
import gradio as gr, base64, io, glob, os, unicodedata
from PIL import Image, ImageDraw

# ── 기능별 예시 질문 (여기만 고치면 버튼이 바뀌어요) ──
QUESTION_GROUPS = [
    ('🍽️', '식사 방법', '어떻게, 뭘 먹으면 좋을까?', [
        '외식할 때 뭘 조심해야 해?',
        '중식집 가는데 뭐 시키는 게 나아?',
        '탄수화물 적은 국물 요리 추천해줘',
        '짜장면 대신 먹을 만한 거 있어?',
    ]),
    ('📊', '혈당 부담', '이거 먹어도 될까?', [
        '짜장면 먹어도 돼?',
        '라면 먹으면 혈당 많이 올라?',
        '김치찌개 칼로리 얼마야?',
        '떡볶이 반만 먹으면 괜찮아?',
    ]),
    ('🍱', '한 끼 합산', '이렇게 먹으면 어때?', [
        '점심에 밥, 김치찌개, 계란말이 먹을 거야',
        '저녁에 비빔밥이랑 된장국 먹으면 어때?',
        '아침으로 토스트랑 우유 먹을 건데 괜찮아?',
        '떡볶이랑 순대 같이 먹으면 혈당 부담 어때?',
    ]),
    ('📝', '하루 기록', '오늘 뭐 먹었지?', [
        '아침에 토스트랑 우유 먹었어. 기록해줘',
        '점심에 비빔밥 먹었어',
        '오늘 먹은 거 정리해줘',
        '오늘 탄수화물 얼마나 먹었어?',
    ]),
]

# ── 이미지 자동 찾기 (한 장만 있어도 OK) ──
nfc = lambda s: unicodedata.normalize('NFC', s).lower()
dirs = [d for d in glob.glob(f'{BASE}/**/', recursive=True) if nfc(d).rstrip('/').endswith('캐릭터')]
CHAR_DIR = dirs[0] if dirs else f'{BASE}/기능2/캐릭터'
imgs = sorted(p for p in glob.glob(f'{CHAR_DIR}/*') if nfc(p).endswith(('.jpg', '.jpeg', '.png', '.webp')))
print('캐릭터 폴더:', CHAR_DIR)
print('찾은 이미지:', [os.path.basename(p) for p in imgs])

def pick(keyword, idx):
    hit = [p for p in imgs if keyword in nfc(os.path.basename(p))]
    if hit: return hit[0]
    if len(imgs) > idx: return imgs[idx]
    return imgs[0] if imgs else None

def circle_avatar(src, dst, size=256):
    im = Image.open(src).convert('RGBA')
    w, h = im.size; s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2)).resize((size, size))
    mask = Image.new('L', (size, size), 0)
    ImageDraw.Draw(mask).ellipse((0, 0, size, size), fill=255)
    im.putalpha(mask); im.save(dst)
    return dst

def to_data_uri(path, max_w=420):
    im = Image.open(path).convert('RGB')
    im.thumbnail((max_w, max_w))
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=88)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()

p_profile, p_hero, p_empty = pick('profile', 0), pick('teacher', 1), pick('banana', 2)
BOT_AVATAR = circle_avatar(p_profile, '/content/bot_avatar.png') if p_profile else None
HERO_TAG  = f'<img src="{to_data_uri(p_hero, 260)}">' if p_hero else ''
EMPTY_TAG = (f'<img src="{to_data_uri(p_empty, 300)}" style="width:150px;border-radius:24px;margin:0 auto 10px">'
             if p_empty else '')

# ── 색감: 갈색 · 크림 · 바나나 노랑 ──
BROWN, BROWN_D, CREAM, BANANA = '#b9805f', '#6b4532', '#fff8ec', '#ffe7a3'

theme = gr.themes.Soft(primary_hue='orange', neutral_hue='stone', radius_size='lg',
                       font=[gr.themes.Font('Pretendard'), 'sans-serif']).set(
    body_background_fill=CREAM, block_background_fill='white', block_border_width='0px',
    block_shadow='0 6px 24px rgba(107, 69, 50, 0.10)',
    button_primary_background_fill=BROWN, button_primary_background_fill_hover=BROWN_D,
    button_primary_text_color='white', body_text_color=BROWN_D)

css = f'''
@import url('https://cdn.jsdelivr.net/gh/orioncactus/pretendard@v1.3.9/dist/web/static/pretendard.min.css');
.gradio-container, .gradio-container * {{ font-family:'Pretendard', sans-serif !important; }}
.gradio-container {{ width: 100% !important; max-width: 1100px !important; margin: auto !important; }}
.gradio-container > .main, .gradio-container .wrap, .gradio-container .contain {{ width: 100% !important; max-width: 100% !important; }}
#hero {{ display:flex; align-items:center; gap:28px; padding:22px 32px; border-radius:28px;
         background: linear-gradient(135deg, {BANANA} 0%, #fff3d1 100%); margin-bottom: 8px; }}
#hero img {{ height:170px; border-radius:22px; }}
#hero h1 {{ font-weight:800; font-size:2.6em; color:{BROWN_D}; margin:0 0 6px 0; letter-spacing:-0.02em; }}
#hero p  {{ color:{BROWN_D}; opacity:.85; margin:2px 0; font-size:1.05em; }}
.message.bot  {{ background:#fffaf0 !important; border:2px solid {BANANA} !important;
                 border-radius:22px 22px 22px 6px !important; }}
.message.bot, .message.bot * {{ color:{BROWN_D} !important; }}   /* 다크 모드에서도 글씨가 보이게 */
.message.user {{ background:{BROWN} !important; color:white !important; border:none !important;
                 border-radius:22px 22px 6px 22px !important; }}
.message.user * {{ color:white !important; }}
.avatar-container {{ width:52px !important; height:52px !important; min-width:52px !important; }}
.avatar-container img {{ width:100% !important; height:100% !important; border-radius:50% !important; border:2px solid {BANANA}; }}
#msg, #msg > *, #msg label, #msg .input-container, #msg .wrap {{ background:transparent !important;
      border:none !important; box-shadow:none !important; padding:0 !important; }}
#msg textarea {{ background:white !important; border-radius:20px !important; border:2px solid {BANANA} !important;
      padding:10px 16px !important; box-shadow:none !important; }}
/* 기능별 질문 열 */
.qcol {{ background:white; border:2px solid {BANANA} !important; border-radius:24px !important;
         padding:16px 14px 14px !important; }}
.qhead {{ text-align:center; margin-bottom:4px; }}
.qhead .t {{ font-weight:700; font-size:1.3em; color:{BROWN_D}; letter-spacing:-0.01em; }}
.qhead .s {{ font-size:.88em; color:{BROWN}; }}
.qbtn {{ font-size:.95em !important; padding:8px 12px !important; background:{CREAM} !important; border:1.5px dashed {BROWN} !important; color:{BROWN_D} !important;
         border-radius:16px !important; text-align:left !important; justify-content:flex-start !important;
         white-space:normal !important; font-weight:500 !important; box-shadow:none !important; }}
.qbtn:hover {{ background:{BANANA} !important; }}
button.lg {{ border-radius:20px !important; }}
footer {{ display:none !important; }}
'''

# ── 대화 처리 ──
def _text(content):
    # Gradio 버전에 따라 content가 문자열 또는 블록 리스트로 와요
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return ' '.join(c.get('text', '') if isinstance(c, dict) else str(c) for c in content)
    return str(content)

def add_user(msg, history):
    history = list(history or [])
    if msg and msg.strip():
        history.append({'role': 'user', 'content': msg.strip()})
    return '', history

def add_bot(history, log_items):
    # 하루 기록(LOG)은 접속한 사람(세션)마다 따로 보관해요: 실행 전에 이 사람의 기록을 넣고, 끝나면 되돌려 받아요
    history = list(history or [])
    if not history or history[-1]['role'] != 'user':
        return history, log_items
    LOG.items[:] = list(log_items or [])
    prior = [{'role': h['role'], 'content': _text(h['content'])} for h in history[:-1]][-6:]
    answer = chat(_text(history[-1]['content']), history=prior)['answer']
    history.append({'role': 'assistant', 'content': answer})
    return history, list(LOG.items)

new_gradio = int(gr.__version__.split('.')[0]) >= 6   # Gradio 6부터 theme/css는 launch()에
style = {'theme': theme, 'css': css}
chat_kwargs = {} if new_gradio else {'type': 'messages'}

with gr.Blocks(**({} if new_gradio else style), title='혈당코치 우땅이', fill_width=True) as demo:
    gr.HTML(f'''
    <div id="hero">
      {HERO_TAG}
      <div>
        <h1>혈당코치 우땅이</h1>
        <p>안녕하세요, 우땅이예요! 먹기 전에 저한테 물어보세요 🍌</p>
        <p style="font-size:.9em">공식 식사요법 자료와 식약처 영양DB로 답해요</p>
      </div>
    </div>''')

    log_state = gr.State([])     # 이 접속자의 하루 기록

    chatbot = gr.Chatbot(
        height=480, avatar_images=(None, BOT_AVATAR), show_label=False, **chat_kwargs,
        placeholder=f'''<div style="text-align:center">{EMPTY_TAG}
            <div style="font-weight:700;font-size:1.3em;color:{BROWN_D}">우땅이랑 오늘 뭐 먹을지 정해요!</div>
            <div style="color:{BROWN};margin-top:4px">아래 질문을 눌러 보거나 직접 물어보세요 🍌</div></div>''')

    with gr.Row():
        msg = gr.Textbox(placeholder='예: 오늘 저녁에 칼국수 먹으려는데 괜찮을까?', show_label=False,
                         scale=8, elem_id='msg', container=False)
        send = gr.Button('보내기', variant='primary', scale=1)
        clear = gr.Button('새 대화', scale=1)

    with gr.Row(equal_height=True):
        for icon, title, sub, questions in QUESTION_GROUPS:
            with gr.Column(elem_classes='qcol', min_width=180):
                gr.HTML(f'<div class="qhead"><div class="t">{icon} {title}</div><div class="s">{sub}</div></div>')
                for q in questions:
                    b = gr.Button(q, size='sm', elem_classes='qbtn')
                    b.click(lambda h, q=q: add_user(q, h), inputs=chatbot, outputs=[msg, chatbot]) \
                     .then(add_bot, inputs=[chatbot, log_state], outputs=[chatbot, log_state])

    msg.submit(add_user, [msg, chatbot], [msg, chatbot]).then(add_bot, [chatbot, log_state], [chatbot, log_state])
    send.click(add_user, [msg, chatbot], [msg, chatbot]).then(add_bot, [chatbot, log_state], [chatbot, log_state])
    def reset_all():
        return '', [], []          # 새 대화를 시작하면 이 접속자의 하루 기록도 비워요
    clear.click(reset_all, None, [msg, chatbot, log_state])

    gr.HTML(f'<p style="text-align:center;color:{BROWN};font-size:.85em;margin-top:6px">'
            '약·인슐린 관련 질문은 의료진과 상담하세요 · 혈당 부담 등급은 참고용이에요</p>')

demo.launch(share=True, debug=False, **(style if new_gradio else {}))